# BiasOps Demo — Resume Screening Model

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
import biasops

# --- Load Adult Census Income dataset ---
data = fetch_openml("adult", version=2, as_frame=True)
df = data.data.copy()
df["income"] = (data.target.astype(str) == ">50K").astype(int)
df["gender"] = df["sex"].astype(str)   # map sex → gender

# Numeric features — gender is NOT a training feature
features = ["age", "education-num", "hours-per-week", "capital-gain", "capital-loss"]
df = df[features + ["gender", "income"]].dropna()

X_train, X_test, y_train, y_test = train_test_split(
    df[features + ["gender"]], df["income"],
    test_size=0.3, random_state=42, stratify=df["income"],
)

# --- Train biased GradientBoosting model (no fairness constraints) ---
model = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
model.fit(X_train[features], y_train)
y_pred = pd.Series(model.predict(X_test[features]), index=X_test.index)

# --- BiasOps: expect BLOCK on adverse_impact_ratio (AIR < 0.80) ---
artifact_before = biasops.eval(
    model=model,
    X_test=X_test,
    y_test=y_test,
    y_pred=y_pred,
    policies=["eeoc_title7_hiring_disparate_impact"],
    protected_cols=["gender"],
    output_dir="./artifacts",
    warn_only=True,
)

In [ ]:
from fairlearn.postprocessing import ThresholdOptimizer

# --- Mitigate: ThresholdOptimizer + DemographicParity ---
# Adjusts per-group decision thresholds to equalize selection rates
# while maximizing balanced accuracy.
postprocessor = ThresholdOptimizer(
    estimator=GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
    constraints="demographic_parity",
    objective="balanced_accuracy_score",
    predict_method="predict_proba",
)
postprocessor.fit(X_train[features], y_train, sensitive_features=X_train["gender"])
y_pred_fair = pd.Series(
    postprocessor.predict(X_test[features], sensitive_features=X_test["gender"]),
    index=X_test.index,
)

# --- BiasOps: should now show PASS ---
artifact_after = biasops.eval(
    model=postprocessor,
    X_test=X_test,
    y_test=y_test,
    y_pred=y_pred_fair,
    policies=["eeoc_title7_hiring_disparate_impact"],
    protected_cols=["gender"],
    output_dir="./artifacts",
    warn_only=True,
)

# Side-by-side run IDs
print(f"\nBefore (biased):    run_id={artifact_before['run_id']}  status={artifact_before['overall_status']}")
print(f"After  (mitigated): run_id={artifact_after['run_id']}  status={artifact_after['overall_status']}")

In [ ]:
import json

# Load both audit artifacts
with open(f"./artifacts/biasops-run-{artifact_before['run_id']}.json") as f:
    art1 = json.load(f)
with open(f"./artifacts/biasops-run-{artifact_after['run_id']}.json") as f:
    art2 = json.load(f)

# Comparison table
def fmt(v):
    return f"{v:.4f}" if isinstance(v, (int, float)) else str(v)

print(f"\n{'Metric':<35} {'Before':<18} {'After':<18}")
print('\u2500' * 72)
for key, label in [
    ("adverse_impact_ratio",    "Adverse Impact Ratio (AIR)"),
    ("demographic_parity_gap",  "Demographic Parity Gap"),
    ("min_group_selection_rate", "Min Group Selection Rate"),
    ("overall_accuracy",        "Overall Accuracy"),
]:
    v1 = art1["metrics"].get(key, "N/A")
    v2 = art2["metrics"].get(key, "N/A")
    print(f"{label:<35} {fmt(v1):<18} {fmt(v2):<18}")

print('\u2500' * 72)
print(f"{'Overall Status':<35} {art1['overall_status']:<18} {art2['overall_status']:<18}")
print(f"{'BLOCK violations':<35} {art1['block_count']:<18} {art2['block_count']:<18}")
print()
print("Citation: EEOC 29 CFR \u00a71607.4(D) (1978)")